## Building a RAG System with LangChain and FAISS

In [2]:
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (RunnablePassthrough)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

#Langchain specific imports 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
load_dotenv()

True

## Data Ingestion and Parsing

In [3]:
sample_documents = [
    Document(page_content="""
            Artificial Intelligence (AI) is the simulation of human intelligence in machines.
            These systems are designed to think like humans and mimic their actions.
            AI can be categorized into narrow AI and general AI.
            """,
            metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
        ),
    Document(
            page_content="""
            Machine Learning is a subset of AI that enables systems to learn from data.
            Instead of being explicitly programmed, ML algorithms find patterns in data.
            Common types include supervised, unsupervised, and reinforcement learning.
            """,
            metadata={"source": "ML Basics", "page": 1, "topic": "ML"} 
        ),
        Document(
            page_content="""
            Deep Learning is a subset of machine learning based on artificial neural networks.
            It uses multiple layers to progressively extract higher-level features from raw input.
            Deep learning has revolutionized computer vision, NLP, and speech recognition.
            """,
            metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
        ),
        Document(
            page_content="""
            Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
            It combines computational linguistics with machine learning and deep learning models.
            Applications include chatbots, translation, sentiment analysis, and text summarization.
            """,
            metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
        )
    ]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n            Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n            These systems are designed to think like humans and mimic their actions.\n            AI can be categorized into narrow AI and general AI.\n            '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n            Machine Learning is a subset of AI that enables systems to learn from data.\n            Instead of being explicitly programmed, ML algorithms find patterns in data.\n            Common types include supervised, unsupervised, and reinforcement learning.\n            '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n            Deep Learning is a subset of machine learning based on artificial neural networks.\n            It uses multiple layers to progressively extract higher-level features from raw 

In [4]:
##Text Splitting
text_splitters = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "]
)

##Split the docuuments into chunks
chunks = text_splitters.split_documents(sample_documents)
for i in range(len(chunks)):
    print(chunks[i])

page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
            These systems are designed to think like humans and mimic their actions.
            AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
            Instead of being explicitly programmed, ML algorithms find patterns in data.
            Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}
page_content='Deep Learning is a subset of machine learning based on artificial neural networks.
            It uses multiple layers to progressively extract higher-level features from raw input.
            Deep learning has revolutionized computer vision, NLP, and speech recognition.' metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}
page_conten

In [5]:
print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunks")
print(f"Content:{chunks[0].page_content}")
print(f"Metadata:{chunks[0].metadata}")


Created 4 chunks from 4 documents

Example chunks
Content:Artificial Intelligence (AI) is the simulation of human intelligence in machines.
            These systems are designed to think like humans and mimic their actions.
            AI can be categorized into narrow AI and general AI.
Metadata:{'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [6]:
## loading huggingface embedding models
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
sample_text = "what is Machine learning?"
sample_embeddings = embeddings.embed_query(sample_text)
sample_embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10678.86it/s]


[-0.019954558461904526,
 0.009878003038465977,
 0.010249651037156582,
 0.029553711414337158,
 0.027186403051018715,
 -0.01929655857384205,
 -0.024129625409841537,
 -0.037735722959041595,
 -0.04105420038104057,
 -0.0014749806141480803,
 -0.07607679814100266,
 0.03687206655740738,
 0.05372380092740059,
 -0.07053521275520325,
 -0.08609342575073242,
 0.021006831899285316,
 -0.04226336628198624,
 0.02844891883432865,
 -0.03837532550096512,
 -0.06591915339231491,
 0.013949612155556679,
 0.012275389395654202,
 -0.06821971386671066,
 0.028625069186091423,
 0.014761177822947502,
 0.03223147988319397,
 0.008694288320839405,
 0.013062099926173687,
 -0.015726706013083458,
 0.01889656111598015,
 0.020023876801133156,
 -0.039297088980674744,
 0.015647143125534058,
 0.033398013561964035,
 -0.07339106500148773,
 0.059357818216085434,
 -0.012385823763906956,
 0.02458462119102478,
 0.020107083022594452,
 -0.020254511386156082,
 -0.05024769529700279,
 -0.09044112265110016,
 -0.015374395996332169,
 -0.003

In [7]:
texts = ["AI","machine Learning","Deep Learning","Neural Network"]
batch_embeddings = embeddings.embed_documents(texts)
for i in range(len(batch_embeddings)):
    print(batch_embeddings[i])

[-0.03653930500149727, -0.01516442559659481, 0.01643247716128826, 0.010568788275122643, 0.006010600831359625, -0.018473222851753235, 0.08546529710292816, 0.020968450233340263, 0.027815278619527817, 0.012431635521352291, -0.02937653474509716, -0.031135324388742447, 0.034912511706352234, -0.018150843679904938, -0.06498483568429947, 0.051682498306035995, -0.019606154412031174, -0.015734154731035233, -0.13371680676937103, -0.09645990282297134, -0.02547178789973259, -0.001489552785642445, -0.006349349394440651, -0.025820767506957054, -0.027371780946850777, 0.12268999218940735, -0.007792397867888212, -0.03852269798517227, 0.014383583329617977, -0.09218421578407288, 0.00869578868150711, 0.0026133349165320396, 0.0910346508026123, -0.03031357191503048, -0.09604641795158386, 0.022289130836725235, -0.09024307876825333, -0.03294738754630089, 0.07158330827951431, -0.008893145248293877, -0.025708964094519615, -0.07913964241743088, 0.01453041285276413, -0.07420431822538376, 0.08045008033514023, 0.078

In [8]:
## Compare Embeddings using Cosine Similarity

def compare_embeddings(text1:str, text2:str):
    """Compare semantic similarity of 2 texts using embeddings"""

    emb1 = np.array(embeddings.embed_query(text1))
    emb2 = np.array(embeddings.embed_query(text2))

    ##Caluculate the similarity score

    similarity= np.dot(emb1,emb2)/(np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity



In [9]:
## Test Semantic Similarity
print("\nSemantic Similarity Examples:")
print(f"'AI' vs 'Artificical Intelligence' :{compare_embeddings('AI','Artificial Intellegence'):.3f}") 


Semantic Similarity Examples:
'AI' vs 'Artificical Intelligence' :0.675


In [10]:
print(f"AI vs Pizza : {compare_embeddings('AI','Pizza'):.3f}")

AI vs Pizza : 0.257


In [11]:
print(f"Machine Learning vs ML : {compare_embeddings('Machine Learning','ML'):.3f}")


Machine Learning vs ML : 0.373


## Cerate FAISS Vector Store

In [12]:
vectorstore = FAISS.from_documents(
    documents = chunks,
    embedding = embeddings
)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 4 vectors


In [13]:
vectorstore

In [14]:
##Save vector for later use
vectorstore.save_local("faiss_index")
print("Vectorstore saved to faiss_index directory")

Vectorstore saved to faiss_index directory


In [15]:
##load the vecotrstore from disk
loaded_vectorstore=FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization = True
)
print(f"Loaded Vecotr store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded Vecotr store contains 4 vectors


In [16]:
#Similarity Search
query = "What is deep learning?"
results = vectorstore.similarity_search(query, k = 3)
print(results)

[Document(id='7153215a-79d3-4551-8291-af014d2f41ed', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n            It uses multiple layers to progressively extract higher-level features from raw input.\n            Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='bb8ea130-8fe0-4d7b-b747-01ffcac36966', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n            Instead of being explicitly programmed, ML algorithms find patterns in data.\n            Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='9d5c9d0b-5adc-4890-bd3c-45db7e736375', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers under

In [17]:
print(f"Query:{query}\n")
print("To; 3 similar chunks:")
for i,doc in enumerate(results):
    print(f"\n{i+1}. Source:{doc.metadata['source']}")
    print(f" Content:{doc.page_content[:200]}...")

Query:What is deep learning?

To; 3 similar chunks:

1. Source:Deep Learning
 Content:Deep Learning is a subset of machine learning based on artificial neural networks.
            It uses multiple layers to progressively extract higher-level features from raw input.
            Deep l...

2. Source:ML Basics
 Content:Machine Learning is a subset of AI that enables systems to learn from data.
            Instead of being explicitly programmed, ML algorithms find patterns in data.
            Common types include su...

3. Source:NLP Overview
 Content:Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
            It combines computational linguistics with machine learning and deep learning models.
  ...


In [18]:
## similarity search with score
results_with_scores = vectorstore.similarity_search_with_score(query, k =3)
print(f"\n\n Similarity Search with Scores:")
for doc, score in results_with_scores:
    print(f"\nScore:{score:.3f}")
    print(f"Source:{doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")




 Similarity Search with Scores:

Score:0.394
Source:Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
            It us...

Score:1.007
Source:ML Basics
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
            Instead of b...

Score:1.112
Source:NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
...


In [19]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n            These systems are designed to think like humans and mimic their actions.\n            AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n            Instead of being explicitly programmed, ML algorithms find patterns in data.\n            Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n            It uses multiple layers to progressively extract higher-level features from raw input.\n            Deep learning has revolutionized computer vision

In [20]:
# Search with metadata filtering
filter_dict = {"topic":"ML"}
filtered_results = vectorstore.similarity_search(
    query,
    k=3,
    filter = filter_dict
)
print(filtered_results)

[Document(id='bb8ea130-8fe0-4d7b-b747-01ffcac36966', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n            Instead of being explicitly programmed, ML algorithms find patterns in data.\n            Common types include supervised, unsupervised, and reinforcement learning.')]


In [21]:
len(filtered_results)

1

## Build RAG Chain with LCEL

In [39]:
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_model(model = "groq:openai/gpt-oss-120b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x133358550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x133358f50>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [40]:
#1. Simple RACG CHain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answere the question based only on the following context:
Context:{context}

Question:{question}

Answer:
""")

In [41]:
#Basic retriever
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":3}
)

In [42]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12bd67230>, search_kwargs={'k': 3})

In [43]:
from typing import List

#Format document for the prompt
def format_docs(docs:List[Document])->str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source','unknown')
        formatted.append(f"Document {i+1} (Source: {source}): \n{doc.page_content}")
    return "\n\n".join(formatted)

In [60]:
simple_rag_chain=({
    "context":retriever | format_docs, "question":RunnablePassthrough()
    }
    |simple_prompt
    |llm
    |StrOutputParser()
    )

In [61]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12bd67230>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answere the question based only on the following context:\nContext:{context}\n\nQuestion:{question}\n\nAnswer:\n'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'te

In [62]:
#Conversation Rag Chain 
coversational_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful AI assistant. Use the provided context to answer questions"),
    ("placeholder","{chat_history}"),
    ("human","Context:{context}\n\nQuestion:{input}")
])

In [63]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return(
        RunnablePassthrough.assign(
            context = lambda x: format_docs(retriever.invoke(x["input"])))
        | coversational_prompt
        | llm
        |StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [64]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(...)
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')] | typing.Annotated[langchain_cor

In [70]:
### Streaming RAG Chain
streaming_rag_chain = (
    {"context":retriever | format_docs, "question":RunnablePassthrough()}
    |simple_prompt
    |llm
)
print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [71]:
##test function for different chain types:
def test_rag_chain(question:str):
    """Test all RAG chain variants"""
    print(f"Question:{question}")
    print("="*80)
    # 1. Simple RAG Chain
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    #2. Coversational RAG Chain
    print(f"2. Conversational RAG Chain")
    answer = conversational_rag.invoke({
        "input":question,
        "chat_history":[]
    })
    print(f"Answer: {answer}")

    print(f"\n3. Streaming RAG Chain")
    answer = streaming_rag_chain.invoke(question)
    print(f"Answer: ",end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end = "", flush = True)
    print()

In [72]:
test_rag_chain("What is the difference between AI and MAchine learning?")

Question:What is the difference between AI and MAchine learning?

1. Simple RAG Chain:
Answer: Artificial Intelligence (AI) is the broad field that aims to simulate human intelligence in machines, encompassing any system that can think or act like a human (including narrow and general AI).  

Machine Learning (ML) is a **subset of AI**. It specifically refers to techniques that let systems learn from data instead of being explicitly programmed, using algorithms that discover patterns and can be supervised, unsupervised, or reinforcement‑learning based.  

So, AI is the overall concept of intelligent machines, while ML is one particular way—learning from data—by which AI can be achieved.
2. Conversational RAG Chain
Answer: **Artificial Intelligence (AI)** is the broad discipline that aims to create machines that can simulate human intelligence—thinking, reasoning, and acting like humans. It includes any approach that tries to give computers intelligent behavior, whether through rule‑bas

In [74]:
test_questions = ["What is the difference between AI and Machine Learning",
                  "Explain Deep Learning in simple terms.",
                  "How does NLP work?"
]
for questions in test_questions:
    print("\n" + "=" *80 +"\n")
    test_rag_chain(questions)



Question:What is the difference between AI and Machine Learning

1. Simple RAG Chain:
Answer: **Answer:**  

Artificial Intelligence (AI) is the broad field that aims to simulate human intelligence in machines, enabling them to think, reason, and act like humans. It encompasses any technique that allows a system to exhibit intelligent behavior, ranging from rule‑based expert systems to advanced robotics.

Machine Learning (ML) is a **subset of AI**. Rather than being explicitly programmed with fixed rules, ML algorithms learn from data: they automatically discover patterns and improve their performance as they are exposed to more examples. Supervised, unsupervised, and reinforcement learning are common types of ML.

So, the key difference is that AI refers to the overall goal of creating intelligent machines, while ML is a specific approach within AI that achieves intelligence by learning from data.
2. Conversational RAG Chain
Answer: **Artificial Intelligence (AI)** and **Machine Le

In [75]:
# Conversational Example
print("\n3. Conversational Rag Example: ")
chat_history = []
#First question
q1 = "What is Machine learning?"
a1= conversational_rag.invoke({
    "input":q1,
    "chat_history":chat_history
})

print(f"Q1:{q1}")
print(f"A1: {a1}")


3. Conversational Rag Example: 
Q1:What is Machine learning?
A1: Machine learning (ML) is a branch of artificial intelligence that lets computers improve their performance on a task by learning from data instead of being explicitly programmed.  ML algorithms automatically discover patterns and relationships within the data, enabling the system to make predictions or decisions.  The main categories of machine‑learning approaches include supervised learning, unsupervised learning, and reinforcement learning.


In [76]:
#Update History
chat_history.extend([
    HumanMessage(content = q1),
    AIMessage(content=a1)
])

In [77]:
#Followup Question
q2 = "How it is differenet from traditional programming?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history":chat_history
})
print(f"\nQ2:{q2}")
print(f"A2: {a2}")


Q2:How it is differenet from traditional programming?
A2: **Traditional programming vs. Machine‑Learning‑based programming**

| Aspect | Traditional (rule‑based) programming | Machine‑learning (ML) programming |
|--------|--------------------------------------|-----------------------------------|
| **How the system works** | The developer writes explicit **if‑then‑else** rules that dictate exactly what the program should do for every possible input. | The developer supplies an **algorithm** and a **training dataset**; the algorithm automatically discovers patterns and builds a model that makes predictions or decisions. |
| **Source of knowledge** | Knowledge is encoded by humans (e.g., algorithms, heuristics, business rules). | Knowledge is extracted from data; the model “learns” the rules implicitly. |
| **Handling complexity** | Becomes unwieldy when the problem space is huge or poorly understood (e.g., natural language, image recognition). | Well‑suited for high‑dimensional, noisy,